# Google Cloud Compute Engine 인스턴스 생성 및 리전별 비용 분석

이 노트북은 Google Cloud CLI(`gcloud`)를 사용하여 다음 작업을 수행합니다:
1. **동일 사양(`e2-medium` + 10GB `pd-balanced`) 기준 전 세계 리전별 비용 비교 및 최저가 Top 3 리전 분석** (순수 파이썬으로 동작, 외부 패키지 불필요)
2. `e2-medium` 사양의 Compute Engine 인스턴스 생성 (`instance-20260914-055112`)
3. Cloud Ops Agent 정책 설정 파일 (`config.yaml`) 생성 및 정책 적용
4. 인스턴스 실행 상태 및 배포 결과 확인
5. 리소스 정리 (삭제 코드)
6. 퇴근 전 잔여 과금 리소스 점검 하네스 실행

## 0. 사전 확인 (계정 및 프로젝트)
현재 설정된 gcloud 활성 계정과 프로젝트가 올바른지 확인합니다.

In [ ]:
!gcloud config list

## 1. 동일 사양 기준 전 세계 리전별 비용 비교 및 최저가 Top 3 분석

현재 구성 사양:
- **머신 유형**: `e2-medium` (2 vCPU, 4GB RAM)
- **부팅 디스크**: 10GB `pd-balanced` (균형 있는 영구 디스크)
- **프로비저닝 모델**: `STANDARD` (On-Demand)
- **OS**: Debian 13 (무료 오픈소스 라이선스)

*참고: 이 코드는 pandas 등 외부 라이브러리 설치 없이 **파이썬 기본 표준 라이브러리**만으로 모든 환경에서 오류 없이 바로 실행됩니다.*

In [ ]:
# GCP 리전별 공식 온디맨드 단가 데이터 (e2-medium 및 pd-balanced 10GB 기준)
# 외부 패키지(pandas) 설치 없이 표준 파이썬으로 동작합니다.
data = [
    {'region': 'us-central1', 'location': '아이오와 (미국 중부)', 'vm_hourly': 0.033604, 'disk_gb_monthly': 0.10},
    {'region': 'us-east1', 'location': '사우스캐롤라이나 (미국 동부)', 'vm_hourly': 0.033604, 'disk_gb_monthly': 0.10},
    {'region': 'us-west1', 'location': '오레곤 (미국 서부)', 'vm_hourly': 0.033604, 'disk_gb_monthly': 0.10},
    {'region': 'us-east4', 'location': '북버지니아 (미국 동부)', 'vm_hourly': 0.036964, 'disk_gb_monthly': 0.11},
    {'region': 'us-west4', 'location': '라스베이거스 (미국 서부)', 'vm_hourly': 0.036964, 'disk_gb_monthly': 0.11},
    {'region': 'europe-west1', 'location': '벨기에 (유럽 서부)', 'vm_hourly': 0.037000, 'disk_gb_monthly': 0.11},
    {'region': 'europe-north1', 'location': '핀란드 (유럽 북부)', 'vm_hourly': 0.037000, 'disk_gb_monthly': 0.11},
    {'region': 'asia-east1', 'location': '대만 (아시아 동부)', 'vm_hourly': 0.040324, 'disk_gb_monthly': 0.11},
    {'region': 'asia-northeast3', 'location': '서울 (대한민국)', 'vm_hourly': 0.040400, 'disk_gb_monthly': 0.12},
    {'region': 'asia-northeast1', 'location': '도쿄 (일본)', 'vm_hourly': 0.040400, 'disk_gb_monthly': 0.12},
    {'region': 'southamerica-east1', 'location': '상파울루 (남미)', 'vm_hourly': 0.052086, 'disk_gb_monthly': 0.15}
]

# 730시간 기준 월간 및 시간당 비용 계산
for item in data:
    item['vm_monthly'] = round(item['vm_hourly'] * 730, 2)
    item['disk_monthly'] = round(item['disk_gb_monthly'] * 10, 2)
    item['total_monthly'] = round(item['vm_monthly'] + item['disk_monthly'], 2)
    item['total_hourly'] = round(item['total_monthly'] / 730, 4)

# 최저 비용 순 오름차순 정렬
sorted_data = sorted(data, key=lambda x: x['total_monthly'])

print('=' * 82)
print('[e2-medium + 10GB pd-balanced] 전 세계 최저가 리전 TOP 3')
print('=' * 82)
for rank, row in enumerate(sorted_data[:3], 1):
    print(f"{rank}위: {row['region']:<14} | 위치: {row['location']:<16} | 시간당: ${row['total_hourly']:.4f} | 예상 월 비용: ${row['total_monthly']:.2f}")
print('=' * 82)

# 서울 리전과의 절감액 비교
seoul = next(x for x in data if x['region'] == 'asia-northeast3')
cheapest = sorted_data[0]
diff = round(seoul['total_monthly'] - cheapest['total_monthly'], 2)
pct = round((diff / seoul['total_monthly']) * 100, 1)

print(f"\n[비용 절감 팁]")
print(f"- 국내(서울 asia-northeast3) 리전: 월 ${seoul['total_monthly']:.2f}")
print(f"- 최저가({cheapest['region']}) 리전 선택 시: 월 ${cheapest['total_monthly']:.2f} (월 ${diff:.2f} 절감, 약 {pct}% 저렴!)")

print(f"\n[전체 리전별 상세 비용 비교표]")
print(f"{'순위':^4} | {'리전(Region)':<16} | {'위치':<18} | {'시간당 요금':>10} | {'예상 월 비용':>10} | {'VM 비용':>8} | {'디스크':>6}")
print('-' * 86)
for rank, row in enumerate(sorted_data, 1):
    print(f"{rank:^4} | {row['region']:<16} | {row['location']:<18} | ${row['total_hourly']:>9.4f} | ${row['total_monthly']:>9.2f} | ${row['vm_monthly']:>7.2f} | ${row['disk_monthly']:>5.2f}")

## 2. 전체 명령어 한 번에 실행 (원본 스크립트)
요청하신 원본 bash 명령어를 그대로 실행하는 셀입니다.
(Linux/macOS 또는 Bash 환경 권장)

In [ ]:
!gcloud compute instances create instance-20260915-143200 \
    --project=iceu-songpa23 \
    --zone=us-central1-a \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=695113814332-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-055112,disk-resource-policy=projects/iceu-songpa23/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any \
&& \
printf 'agentsRule:\n  packageState: installed\n  version: latest\ninstanceFilter:\n  inclusionLabels:\n  - labels:\n      goog-ops-agent-policy: v2-template-1-7-0\n' > config.yaml \
&& \
gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa23 \
    --zone=us-central1-a \
    --file=config.yaml

## 3. 단계별 실행 (Windows 및 모든 주피터 환경 권장)
Windows 환경의 주피터 노트북에서는 `\` 줄바꿈이나 `printf`가 기본 셸에서 오류를 일으킬 수 있으므로,
아래와 같이 각 단계를 분리하여 실행하면 모든 OS에서 안전하게 실행됩니다.

### Step 1: Compute Engine 인스턴스 생성
인스턴스 생성을 요청합니다.

In [3]:
# Python subprocess로 실행하여 OS에 관계없이 안전하게 실행
import subprocess
import sys

cmd = [
    "gcloud", "compute", "instances", "create", "instance-20260914-055112",
    "--project=iceu-songpa23",
    "--zone=us-central1-a",
    "--machine-type=e2-medium",
    "--network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default",
    "--metadata=enable-osconfig=TRUE",
    "--maintenance-policy=MIGRATE",
    "--provisioning-model=STANDARD",
    "--service-account=695113814332-compute@developer.gserviceaccount.com",
    "--scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append",
    "--create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-055112,disk-resource-policy=projects/iceu-songpa23/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced",
    "--no-shielded-secure-boot",
    "--shielded-vtpm",
    "--shielded-integrity-monitoring",
    "--labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud",
    "--reservation-affinity=any"
]

result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
print(result.stdout)
if result.stderr:
    print("[STDERR]", result.stderr, file=sys.stderr)

[STDERR] ERROR: (gcloud.compute.instances.create) Could not fetch resource:
 - The resource 'projects/iceu-songpa23/zones/us-central1-a/instances/instance-20260914-055112' already exists




### Step 2: Ops Agent 정책 설정 파일 (`config.yaml`) 생성
주피터 노트북 내장 파일 작성 기능(`%%writefile`)으로 `config.yaml`을 생성합니다.

In [ ]:
%%writefile config.yaml
agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0


### Step 3: Ops Agent 정책 등록 및 적용

In [ ]:
!gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a --project=iceu-songpa23 --zone=us-central1-a --file=config.yaml

## 4. 인스턴스 및 정책 상태 확인

In [4]:
!gcloud compute instances list --project=iceu-songpa23

NAME                      ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
instance-20260914-055112  us-central1-a  e2-medium                  10.128.0.3   35.255.247.107  RUNNING


## 5. 리소스 정리 (과금 방지를 위한 삭제 코드)
실습이 끝난 후 인스턴스와 정책을 정리하려면 아래 셀을 실행하세요.

In [5]:
# 1. 인스턴스 삭제
!gcloud compute instances delete instance-20260914-055112 --zone=us-central1-a --project=iceu-songpa23 --quiet

# 2. Ops Agent 정책 삭제 (필요시)
#!gcloud compute instances ops-agents policies delete goog-ops-agent-v2-template-1-7-0-us-central1-a --zone=us-central1-a --project=iceu-songpa23 --quiet

Deleted [https://www.googleapis.com/compute/v1/projects/iceu-songpa23/zones/us-central1-a/instances/instance-20260914-055112].


## 6. 퇴근 전 잔여 과금 리소스 점검 하네스 실행
혹시 인스턴스나 디스크, 고정 IP 등이 미삭제 상태로 남아있는지 종합 검사합니다.

In [8]:
!python check_gcp_resources.py

 [GCP 과금 방지 잔여 리소스 점검 하네스 (Audit Harness)]
▶ 대상 프로젝트 : iceu-songpa23
▶ 활성 계정     : songpa23@iceu.kr
----------------------------------------------------------------------
• VM 인스턴스                  조회 중... [정리완료] 0개
• 영구 디스크 (Disks)           조회 중... [정리완료] 0개
• 고정/외부 IP 주소              조회 중... [정리완료] 0개
• 디스크 스냅샷                  조회 중... [정리완료] 0개
• 커스텀 머신 이미지               조회 중... [정리완료] 0개
• 부하 분산기 (전달 규칙)           조회 중... [정리완료] 0개
• Cloud NAT 게이트웨이          조회 중... [정리완료] 0개
• Cloud Storage 버킷         조회 중... [정리완료] 0개

 [종합 진단 결과 및 조치 가이드]
[안전] 현재 프로젝트에 익일 요금을 발생시키는 핵심 과금 리소스가 없습니다!
  안심하고 업무를 종료하셔도 됩니다.
